<a href="https://colab.research.google.com/github/adriansivina-droid/scraper-mef/blob/main/HIDROLOGIA/SENAMHI_Rios.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Nivel y caudal de los ríos · SENAMHI

Hidrograma **diario** de las estaciones de la página «Monitoreo Hidrológico» de SENAMHI (nivel en m.s.n.m y caudal en m³/s) y la pestaña **Información Mensual**.

1. Ejecuta el **Bloque 1** (una vez por sesión).
2. Elige en el **Bloque 2**.
3. Ejecuta el **Bloque 3**: descarga, controla y guarda en Drive (`SENAMHI_HIDROLOGIA/DATA` y `CONTROL`).

Contexto del proyecto: `HIDROLOGIA/senamhi_rios.md`.

In [ ]:
# ═══ BLOQUE 1 · INSTALACIÓN Y FUNCIONES (ejecutar una vez por sesión) ═══
!pip install requests pandas ipywidgets google-api-python-client --quiet
import os, re, json, time, unicodedata
from datetime import date, datetime, timezone, timedelta
import requests
import pandas as pd
import ipywidgets as w
from IPython.display import display
print("✅ Librerías listas")

# ── Direcciones de SENAMHI (vistas en el navegador: página «Monitoreo Hidrológico») ──
BASE = "https://www.senamhi.gob.pe"
URL_MAPA = BASE + "/mapas/mapa-monitoreohidro/mapa-monitoreo.php"
URL_ESTACIONES = BASE + "/mapas/mapa-monitoreohidro/include/ajaxEstaciones.php"    # lista de estaciones
URL_IDESEP = BASE + "/mapas/mapa-monitoreohidro/include/ajaxIdesep.php"            # ficha de la estación (vía GeoServer)
URL_GRAFICA = BASE + "/mapas/mapa-monitoreohidro/include/mnt-grafica-new.php"      # hidrograma (horario / diario)
URL_MENSUAL = BASE + "/include/ajax-informacion-mensual.php"                       # pestaña «Información Mensual»
URL_GEOSERVER = "https://idesep.senamhi.gob.pe/geoserver/g_dbphisis"
CAPA = "g_dbphisis:estaciones_monitoreo"
BBOX_PERU = "-9602936.74,-2128006.87,-7090910.24,73379.55"   # EPSG:3857, cubre todo el Perú

CAB = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) "
                     "Chrome/124.0 Safari/537.36",
       "Accept-Language": "es-PE,es;q=0.9",
       "Referer": URL_MAPA,
       "X-Requested-With": "XMLHttpRequest"}
http = requests.Session()
http.headers.update(CAB)
PAUSA = 1.0          # segundos entre consultas


def pedir(metodo, url, intentos=4, **kw):
    """GET/POST con reintentos (2, 4, 8 s). Devuelve el texto o None."""
    for i in range(1, intentos + 1):
        try:
            r = http.request(metodo, url, timeout=60, **kw)
            if r.status_code == 200:
                time.sleep(PAUSA)
                return r.text
            err = f"HTTP {r.status_code}"
        except requests.RequestException as e:
            err = str(e)[:120]
        if i < intentos:
            print(f"   ↻ {url.rsplit('/', 1)[-1]} reintento {i}/{intentos - 1}: {err}")
            time.sleep(2 ** i)
    return None


def sin_tildes(t):
    return "".join(c for c in unicodedata.normalize("NFKD", str(t)) if not unicodedata.combining(c)).upper().strip()


def num(v):
    """Número o None (SENAMHI usa -999 para «sin dato»)."""
    if v is None or v == "":
        return None
    try:
        v = float(v)
    except (TypeError, ValueError):
        return None
    return None if v <= -999 else round(v, 4)


def fecha_ms(ms):
    return datetime.fromtimestamp(ms / 1000, tz=timezone.utc).date()


def un_año_antes(d):
    try:
        return d.replace(year=d.year - 1)
    except ValueError:            # 29/02 → no existe el año anterior
        return None


def año_hidrologico(d):
    """El año hidrológico de SENAMHI va de setiembre a agosto; se nombra por el año en que empieza."""
    return d.year if d.month >= 9 else d.year - 1


# ── 1. Lista de estaciones ────────────────────────────────────────────────────
def leer_estaciones():
    t = pedir("POST", URL_ESTACIONES, files={"cod_zonal": (None, "-1")})
    if not t:
        return []
    return json.loads(t)


def leer_geoserver():
    """Atributos y coordenadas de todas las estaciones (WFS del GeoServer de SENAMHI). Si falla, devuelve {}."""
    t = pedir("GET", URL_GEOSERVER + "/ows", intentos=2,
              params={"service": "WFS", "version": "1.0.0", "request": "GetFeature", "typeName": CAPA,
                      "outputFormat": "application/json", "srsName": "EPSG:4326"})
    out = {}
    try:
        for f in json.loads(t)["features"]:
            p = dict(f.get("properties") or {})
            g = f.get("geometry") or {}
            if g.get("type") == "Point":
                p["LON"], p["LAT"] = g["coordinates"][:2]
            cod = str(p.get("cod_estacion") or p.get("codEstacion") or p.get("id") or
                      str(f.get("id", "")).split(".")[-1])
            out[cod] = p
    except Exception:
        pass
    return out


def ficha_estacion(cod):
    """Lo mismo que pide el mapa al hacer clic: {"id","fecha_hora","variable","variable_opcion"} o None."""
    url = (f"https://idesep.senamhi.gob.pe:443/geoserver/g_dbphisis/wms?service=WMS&request=GetFeatureInfo"
           f"&version=1.1.1&layers=g_dbphisis%3Aestaciones_monitoreo&styles=&format=image%2Fpng&transparent=true"
           f"&cql_filter=id%20IN%20('estaciones_monitoreo.{cod}')&info_format=application%2Fjson"
           f"&width=1&height=1&srs=EPSG%3A3857&bbox={BBOX_PERU}"
           f"&query_layers=g_dbphisis%3Aestaciones_monitoreo&X=0&Y=0")
    t = pedir("POST", URL_IDESEP, intentos=2, files={"url": (None, url)})
    try:
        d = json.loads(t)
        return d if isinstance(d, dict) and d.get("id") else None
    except Exception:
        return None


# ── 2. Hidrograma ─────────────────────────────────────────────────────────────
def leer_var_js(html, nombre):
    """Extrae el objeto JSON de «var <nombre> = {...};» del HTML del hidrograma."""
    m = re.search(r"var\s+" + nombre + r"\s*=\s*", html or "")
    if not m:
        return None
    try:
        return json.JSONDecoder().raw_decode(html, m.end())[0]
    except ValueError:
        return None


def grafica(cod, variable, opcion, fecha_hora, tipo="D"):
    """Hidrograma de una estación: tipo D = diario (año hidrológico), H = horario (últimos días)."""
    html = pedir("POST", URL_GRAFICA, data={"rbtVariable": variable, "fecha_hora": fecha_hora, "btnTipo": tipo,
                                            "variable_opcion": opcion, "id": cod})
    return leer_var_js(html, "data"), html


def filas_diarias(d, variable):
    """Convierte el «Diario» en filas por día. datoActual = año hidrológico pedido; datoAnterior = el anterior."""
    if not d or not d.get("fechaHora"):
        return []
    serie = {k: dict((int(t), v) for t, v in d.get(k) or []) for k in
             ("datoActual", "datoAnterior", "minimo", "maximo", "promHistorico", "rojo", "naranja", "amarillo")}
    base = {"COD_ESTACION": str(d.get("codEstacion", "")), "ESTACION": d.get("nomEstacion", ""),
            "RIO": d.get("nomRio", ""), "VARIABLE": variable, "UNIDAD": d.get("unidad", "")}
    out = []
    for ms in d["fechaHora"]:
        ms = int(ms)
        f = fecha_ms(ms)
        umbrales = {"UMBRAL_ROJO": num(serie["rojo"].get(ms)), "UMBRAL_NARANJA": num(serie["naranja"].get(ms)),
                    "UMBRAL_AMARILLO": num(serie["amarillo"].get(ms)), "PROM_HISTORICO": num(serie["promHistorico"].get(ms))}
        v = num(serie["datoActual"].get(ms))
        if v is not None:
            out.append({**base, "FECHA_D": f, "VALOR": v, "MINIMO": num(serie["minimo"].get(ms)),
                        "MAXIMO": num(serie["maximo"].get(ms)), **umbrales, "SERIE": "ACTUAL"})
        va = num(serie["datoAnterior"].get(ms))
        fa = un_año_antes(f)
        if va is not None and fa is not None:
            out.append({**base, "FECHA_D": fa, "VALOR": va, "MINIMO": None, "MAXIMO": None,
                        "UMBRAL_ROJO": None, "UMBRAL_NARANJA": None, "UMBRAL_AMARILLO": None,
                        "PROM_HISTORICO": None, "SERIE": "ANTERIOR"})
    return out


def inicio_serie(d):
    return fecha_ms(int(d["fechaHora"][0])) if d and d.get("fechaHora") else None


# ── 3. Información Mensual ────────────────────────────────────────────────────
MES_EN = {"JAN": 1, "FEB": 2, "MAR": 3, "APR": 4, "MAY": 5, "JUN": 6, "JUL": 7, "AUG": 8, "SEP": 9, "OCT": 10,
          "NOV": 11, "DEC": 12}


def mes_de_rotulo(t):
    """«JAN-23» → (2023, 1)."""
    m = re.match(r"([A-Z]{3})-(\d{2})$", str(t).strip().upper())
    return (2000 + int(m.group(2)), MES_EN[m.group(1)]) if m and m.group(1) in MES_EN else (None, None)


def leer_mensual(año, mes):
    """Tabla de la pestaña «Información Mensual» para un mes. Devuelve (año, mes) realmente servido y filas."""
    t = pedir("POST", URL_MENSUAL, files={"fecha": (None, f"{año}-{mes:02d}")})
    try:
        d = json.loads(t)
    except Exception:
        return (None, None), []
    filas, servido = [], (None, None)
    for bloque in d.get("content") or []:
        info = bloque.get("informacion") or {}
        servido = mes_de_rotulo(info.get("primDescripcion"))
        for x in bloque.get("detalle") or []:
            filas.append({"COD_ESTACION": str(x.get("codEsta", "")), "ESTACION": x.get("nomEsta", ""),
                          "CUERPO_AGUA": x.get("cuerpoAgua", ""), "DEPARTAMENTO": x.get("nomDepa", ""),
                          "ZONA": x.get("nomSector", ""), "REGION_HIDROGRAFICA": x.get("uniHidrografica", ""),
                          "COD_ZONAL": x.get("codZonal", ""), "UNIDAD": x.get("unidad", ""),
                          "VALOR_MES": num(x.get("dato")), "VALOR_MES_ANTERIOR": num(x.get("datoAnt")),
                          "ANOMALIA": num(x.get("datAnomalia")), "UNIDAD_ANOMALIA": x.get("uniAnomalia", ""),
                          "TENDENCIA": x.get("tendencia") or "", "UMBRAL_ROJO": num(x.get("umbralRojo"))})
    return servido, filas

print("✅ Funciones listas")

In [ ]:
# ═══ BLOQUE 2 · SELECTORES (elige y luego ejecuta el Bloque 3) ═══
_años = list(range(date.today().year, 2009, -1))
_ancho = {"description_width": "190px"}
sel_anio_ini = w.Dropdown(options=_años, value=2023, description="Desde el año", style=_ancho)
chk_nivel = w.Checkbox(value=True, description="Nivel (m.s.n.m)")
chk_caudal = w.Checkbox(value=True, description="Caudal (m³/s), donde la estación lo tenga")
chk_mensual = w.Checkbox(value=True, description="Información Mensual (estaciones principales, desde el año elegido)")
txt_estaciones = w.Text(value="", description="Solo estas estaciones", style=_ancho,
                        placeholder="códigos separados por coma (vacío = todas)")
display(w.VBox([
    w.HTML("<b>Período</b>. El hidrograma «Diario» trae un <b>año hidrológico</b> (setiembre–agosto) y el anterior; "
           "para llegar al año elegido se piden años hidrológicos anteriores."),
    sel_anio_ini,
    w.HTML("<b>Variables</b> del hidrograma (una fila por estación, variable y día)."), chk_nivel, chk_caudal,
    w.HTML("La pestaña <b>Información Mensual</b> trae el valor del mes, la anomalía y la tendencia de ~57 estaciones."),
    chk_mensual,
    w.HTML("Para una prueba rápida, escribe 2 o 3 códigos (p. ej. <code>240001, 240111, 240103</code>)."), txt_estaciones,
]))

In [ ]:
# ═══ BLOQUE 3 · DESCARGA, CONTROL Y DRIVE ═══
# Si se corta, vuelve a ejecutarlo: los CSV de DATA se acumulan (cada corrida reemplaza los días que vuelve a leer).

# ── 1. CONFIGURACIÓN ─────────────────────────────────────────────────────────
DRIVE_FOLDER_ID = ""            # DATA. Vacío → se usa/crea «SENAMHI_HIDROLOGIA/DATA» en Mi unidad
DRIVE_FOLDER_CONTROL_ID = ""    # CONTROL. Vacío → se usa/crea «SENAMHI_HIDROLOGIA/CONTROL»
NOMBRE_ESTACIONES = "SENAMHI_ESTACIONES.csv"
NOMBRE_DIARIO = "SENAMHI_rios_diario.csv"
NOMBRE_MENSUAL = "SENAMHI_rios_mensual.csv"
NOMBRE_CONTROL = "CONTROL SENAMHI rios_diario.csv"
NOMBRE_CONTROL_MENSUAL = "CONTROL SENAMHI rios_mensual.csv"
NOMBRE_LOG = "LOG SENAMHI consultas.csv"
ruta = "/content/Senamhi"; os.makedirs(ruta, exist_ok=True)
A_INI = sel_anio_ini.value
DESDE = date(A_INI, 1, 1)
HOY = datetime.now(timezone(timedelta(hours=-5))).date()          # hora de Lima
VARIABLES = [v for v, c in (("NIVEL", chk_nivel), ("CAUDAL", chk_caudal)) if c.value]
SOLO = {c.strip() for c in txt_estaciones.value.split(",") if c.strip()}
HOY_TXT = HOY.strftime("%d/%m/%Y")

# ── 2. GOOGLE DRIVE ──────────────────────────────────────────────────────────
from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload, MediaIoBaseDownload
import logging
auth.authenticate_user()
drive = build("drive", "v3", cache_discovery=False)
logging.getLogger("google_auth_httplib2").setLevel(logging.ERROR)

def archivos_en(folder_id):
    out, token = {}, None
    while True:
        r = drive.files().list(q=f"'{folder_id}' in parents and trashed = false",
                               fields="nextPageToken, files(id,name,mimeType)", pageSize=1000, pageToken=token,
                               supportsAllDrives=True, includeItemsFromAllDrives=True).execute()
        out.update({f["name"]: f["id"] for f in r.get("files", [])})
        token = r.get("nextPageToken")
        if not token:
            return out

def carpeta(nombre, padre="root"):
    q = (f"name = '{nombre}' and '{padre}' in parents and trashed = false "
         f"and mimeType = 'application/vnd.google-apps.folder'")
    r = drive.files().list(q=q, fields="files(id)", supportsAllDrives=True, includeItemsFromAllDrives=True).execute()
    if r.get("files"):
        return r["files"][0]["id"]
    return drive.files().create(body={"name": nombre, "parents": [padre], "mimeType": "application/vnd.google-apps.folder"},
                                fields="id", supportsAllDrives=True).execute()["id"]

def subir(ruta_local, folder_id, existentes, mime="text/csv"):
    nombre = os.path.basename(ruta_local)
    for i in range(1, 6):
        try:
            media = MediaFileUpload(ruta_local, mimetype=mime, resumable=True)
            if nombre in existentes:
                drive.files().update(fileId=existentes[nombre], media_body=media, supportsAllDrives=True).execute()
            else:
                f = drive.files().create(body={"name": nombre, "parents": [folder_id]}, media_body=media,
                                         fields="id", supportsAllDrives=True).execute()
                existentes[nombre] = f["id"]
            return True
        except Exception as e:
            print(f"   ↻ Drive reintento {i}/5: {e}"); time.sleep(2 ** i)
    return False

def bajar_de_drive(file_id, destino):
    with open(destino, "wb") as fh:
        dl = MediaIoBaseDownload(fh, drive.files().get_media(fileId=file_id, supportsAllDrives=True))
        hecho = False
        while not hecho:
            _, hecho = dl.next_chunk()

def leer_previo(nombre):
    """CSV acumulado que ya está en DATA (o un DataFrame vacío)."""
    if nombre not in datos_drive:
        return pd.DataFrame()
    destino = os.path.join(ruta, "previo_" + nombre)
    bajar_de_drive(datos_drive[nombre], destino)
    return pd.read_csv(destino, encoding="utf-8-sig", dtype={"COD_ESTACION": str})

if not DRIVE_FOLDER_ID or not DRIVE_FOLDER_CONTROL_ID:
    _raiz = carpeta("SENAMHI_HIDROLOGIA")
    DRIVE_FOLDER_ID = DRIVE_FOLDER_ID or carpeta("DATA", _raiz)
    DRIVE_FOLDER_CONTROL_ID = DRIVE_FOLDER_CONTROL_ID or carpeta("CONTROL", _raiz)
datos_drive = archivos_en(DRIVE_FOLDER_ID)
control_drive = archivos_en(DRIVE_FOLDER_CONTROL_ID)
print(f"☁️  Drive conectado · DATA {DRIVE_FOLDER_ID} · CONTROL {DRIVE_FOLDER_CONTROL_ID}")

def guardar(df, nombre, folder_id, existentes):
    p = os.path.join(ruta, nombre)
    df.to_csv(p, index=False, encoding="utf-8-sig")
    subir(p, folder_id, existentes)

def con_fecha(df, col="FECHA_D"):
    df = df.copy()
    df["FECHA"] = pd.to_datetime(df[col]).dt.strftime("%d/%m/%Y")
    df["AÑO"] = pd.to_datetime(df[col]).dt.year
    df["MES"] = pd.to_datetime(df[col]).dt.month
    df["DIA"] = pd.to_datetime(df[col]).dt.day
    return df

# ── 3. ESTACIONES ────────────────────────────────────────────────────────────
log = []
lista = leer_estaciones()
if not lista:
    raise RuntimeError("SENAMHI no respondió la lista de estaciones (ajaxEstaciones.php). "
                       "Prueba otra vez en unos minutos o abre la página en el navegador para ver si está caída.")
est = pd.DataFrame(lista).rename(columns={"idEstacion": "ID_ESTACION", "codEstacion": "COD_ESTACION",
                                          "nomEstacion": "ESTACION", "subEstacion": "SUB_ESTACION", "estado": "ESTADO"})
est["COD_ESTACION"] = est["COD_ESTACION"].astype(str)
geo = leer_geoserver()
if geo:
    g = pd.DataFrame.from_dict(geo, orient="index")
    g.columns = ["GEO_" + sin_tildes(c).replace(" ", "_") if c not in ("LAT", "LON") else c for c in g.columns]
    est = est.merge(g, left_on="COD_ESTACION", right_index=True, how="left")
    print(f"🗺️  GeoServer: atributos y coordenadas de {est['LAT'].notna().sum() if 'LAT' in est else 0} estaciones")
else:
    print("ℹ️  El GeoServer no respondió la lista (WFS); se sigue sin coordenadas.")
if SOLO:
    est = est[est["COD_ESTACION"].isin(SOLO)]
print(f"📍 Estaciones a consultar: {len(est)} · variables: {', '.join(VARIABLES) or '—'} · desde {DESDE:%d/%m/%Y}")

# ── 4. HIDROGRAMA DIARIO POR ESTACIÓN ────────────────────────────────────────
# Cada consulta «Diario» trae el año hidrológico de la fecha pedida (datoActual) y el anterior (datoAnterior).
# Para ir hacia atrás se pide el 31/08 del año siguiente: año Y → trae Y y Y-1; luego Y-2 …
AH_HOY, AH_MIN = año_hidrologico(HOY), año_hidrologico(DESDE)
historia_ok = None          # None = aún no se sabe si SENAMHI acepta fechas pasadas; True/False tras las primeras pruebas
pruebas_historia = []
filas, fichas, crudo_guardado = [], {}, False
for k, e in enumerate(est.itertuples(), 1):
    cod = e.COD_ESTACION
    ficha = ficha_estacion(cod) or {}
    fichas[cod] = ficha
    fh_actual = ficha.get("fecha_hora") or HOY.strftime("%Y-%m-%d 06:00:00")
    opcion = ficha.get("variable_opcion") or "N"
    for var in VARIABLES:
        y, n_var = AH_HOY, 0
        while True:
            fh = fh_actual if y == AH_HOY else f"{y + 1}-08-31 06:00:00"
            d, html = grafica(cod, var, opcion, fh)
            ini = inicio_serie(d)
            nuevas = filas_diarias(d, var)
            if d and sin_tildes(d.get("variable", var)) != var:     # pidió caudal y devolvió nivel (o al revés)
                nuevas = []
            esperado = date(y, 9, 1)
            estado = ("SIN RESPUESTA" if html is None else "SIN DATOS" if not nuevas else
                      "OK" if ini == esperado else f"DEVOLVIO OTRO AÑO ({ini:%d/%m/%Y})" if ini else "SIN FECHAS")
            log.append({"COD_ESTACION": cod, "ESTACION": e.ESTACION, "VARIABLE": var, "VARIABLE_OPCION": opcion,
                        "FECHA_HORA_PEDIDA": fh, "INICIO_SERIE": ini.strftime("%d/%m/%Y") if ini else "",
                        "FILAS": len(nuevas), "ESTADO": estado, "FECHA_EXTRACCION": HOY_TXT})
            if html and not nuevas and not crudo_guardado:      # diagnóstico: una respuesta vacía, tal cual
                open(os.path.join(ruta, "SENAMHI_respuesta_vacia.html"), "w", encoding="utf-8").write(html)
                subir(os.path.join(ruta, "SENAMHI_respuesta_vacia.html"), DRIVE_FOLDER_CONTROL_ID, control_drive,
                      mime="text/html")
                crudo_guardado = True
            if estado.startswith("OK"):
                filas += nuevas; n_var += len(nuevas)
            elif y == AH_HOY:
                break                                   # la estación no tiene esta variable
            if y != AH_HOY:                             # ¿SENAMHI acepta fechas pasadas?
                if historia_ok is None:
                    pruebas_historia.append(estado == "OK")
                    if any(pruebas_historia):
                        historia_ok = True
                    elif len(pruebas_historia) >= 3:
                        historia_ok = False
                        print("⚠️ SENAMHI ignora la fecha pedida: el hidrograma solo llega hasta el año hidrológico "
                              f"{AH_HOY - 1}-{AH_HOY}. Para años anteriores queda la Información Mensual.")
                if estado != "OK":
                    break
            if y - 1 <= AH_MIN or historia_ok is False:
                break
            y -= 2
    if k % 10 == 0 or k == len(est):
        print(f"   {k}/{len(est)} estaciones · {len(filas):,} filas")

# ── 5. ACUMULAR Y GUARDAR EL DIARIO ──────────────────────────────────────────
CLAVE = ["COD_ESTACION", "VARIABLE", "FECHA"]
COLS_D = ["COD_ESTACION", "ESTACION", "RIO", "VARIABLE", "UNIDAD", "FECHA", "AÑO", "MES", "DIA", "AÑO_HIDROLOGICO",
          "VALOR", "MINIMO", "MAXIMO", "PROM_HISTORICO", "UMBRAL_ROJO", "UMBRAL_NARANJA", "UMBRAL_AMARILLO",
          "SERIE", "FECHA_EXTRACCION"]
dia = pd.DataFrame(filas)
if len(dia):
    dia = dia[dia["FECHA_D"] >= DESDE]
    dia["AÑO_HIDROLOGICO"] = [año_hidrologico(f) for f in dia["FECHA_D"]]
    dia = con_fecha(dia)
    dia["FECHA_EXTRACCION"] = HOY_TXT
    dia = dia[COLS_D]
previo = leer_previo(NOMBRE_DIARIO)
todo = pd.concat([dia, previo], ignore_index=True) if len(previo) else dia
if len(todo):
    # prioridad: dato del año pedido (ACTUAL) sobre el del año anterior; a igualdad, lo leído hoy
    todo["_p"] = (todo["SERIE"] != "ACTUAL").astype(int)
    todo["_n"] = range(len(todo))
    todo = (todo.sort_values(["_p", "_n"]).drop_duplicates(CLAVE, keep="first")
            .drop(columns=["_p", "_n"]))
    todo["_f"] = pd.to_datetime(todo["FECHA"], format="%d/%m/%Y")
    todo = todo.sort_values(["COD_ESTACION", "VARIABLE", "_f"]).drop(columns="_f")
    guardar(todo, NOMBRE_DIARIO, DRIVE_FOLDER_ID, datos_drive)
    print(f"💾 DATA/{NOMBRE_DIARIO}: {len(todo):,} filas · {todo['COD_ESTACION'].nunique()} estaciones "
          f"(nuevas en esta corrida: {len(dia):,})")

# ficha de estaciones (con la última fecha y las variables que trae cada una)
est["FECHA_HORA_ULTIMA"] = [fichas.get(c, {}).get("fecha_hora", "") for c in est["COD_ESTACION"]]
est["VARIABLE_OPCION"] = [fichas.get(c, {}).get("variable_opcion", "") for c in est["COD_ESTACION"]]
if len(todo):
    rio = todo.drop_duplicates("COD_ESTACION").set_index("COD_ESTACION")["RIO"]
    est["RIO"] = est["COD_ESTACION"].map(rio)
    est["VARIABLES_CON_DATOS"] = est["COD_ESTACION"].map(
        todo.groupby("COD_ESTACION")["VARIABLE"].agg(lambda s: " / ".join(sorted(set(s)))))
est["FECHA_EXTRACCION"] = HOY_TXT
if not SOLO:
    guardar(est, NOMBRE_ESTACIONES, DRIVE_FOLDER_ID, datos_drive)
guardar(pd.DataFrame(log), NOMBRE_LOG, DRIVE_FOLDER_CONTROL_ID, control_drive)

# ── 6. INFORMACIÓN MENSUAL ───────────────────────────────────────────────────
men = pd.DataFrame()
if chk_mensual.value:
    filas_m, ctrl_m = [], []
    a, m = A_INI, 1
    while (a, m) <= (HOY.year, HOY.month):
        servido, fm = leer_mensual(a, m)
        ok = servido == (a, m)
        ctrl_m.append({"AÑO": a, "MES": m, "SERVIDO": f"{servido[1]:02d}/{servido[0]}" if servido[0] else "",
                       "ESTACIONES": len(fm), "ESTADO": "OK" if ok and fm else "SIN DATOS" if not fm else
                       "AÚN NO PUBLICADO (devuelve otro mes)", "FECHA_EXTRACCION": HOY_TXT})
        if ok:
            for f in fm:
                filas_m.append({"AÑO": a, "MES": m, "FECHA": f"01/{m:02d}/{a}", **f, "FECHA_EXTRACCION": HOY_TXT})
        a, m = (a + 1, 1) if m == 12 else (a, m + 1)
    men = pd.DataFrame(filas_m)
    previo_m = leer_previo(NOMBRE_MENSUAL)
    if len(previo_m):
        men = pd.concat([men, previo_m], ignore_index=True).drop_duplicates(["COD_ESTACION", "AÑO", "MES"], keep="first")
    if len(men):
        men = men.sort_values(["COD_ESTACION", "AÑO", "MES"])
        guardar(men, NOMBRE_MENSUAL, DRIVE_FOLDER_ID, datos_drive)
        print(f"💾 DATA/{NOMBRE_MENSUAL}: {len(men):,} filas · {men['COD_ESTACION'].nunique()} estaciones · "
              f"{men['AÑO'].min()}–{men['AÑO'].max()}")
    guardar(pd.DataFrame(ctrl_m), NOMBRE_CONTROL_MENSUAL, DRIVE_FOLDER_CONTROL_ID, control_drive)

# departamento de cada estación: GeoServer (si trae ese atributo) y, si no, la Información Mensual
DEPTO = {}
for c in [c for c in est.columns if "DEPA" in c]:
    DEPTO.update({k: sin_tildes(v) for k, v in zip(est["COD_ESTACION"], est[c]) if isinstance(v, str) and v})
if len(men):
    DEPTO.update({k: sin_tildes(v) for k, v in zip(men["COD_ESTACION"], men["DEPARTAMENTO"])
                  if isinstance(v, str) and v and k not in DEPTO})
if DEPTO and not SOLO:
    est["DEPARTAMENTO"] = est["COD_ESTACION"].map(DEPTO)
    guardar(est, NOMBRE_ESTACIONES, DRIVE_FOLDER_ID, datos_drive)

# ── 7. CONTROL ───────────────────────────────────────────────────────────────
# Por estación y variable: días con dato, huecos y saltos raros; y el cruce con la Información Mensual
# (promedio de los días del mes vs el valor mensual publicado, cuando la unidad es la misma).
ctrl = []
lg = pd.DataFrame(log)
if len(lg):
    for (cod, var), g in lg.groupby(["COD_ESTACION", "VARIABLE"]):
        t = todo[(todo["COD_ESTACION"] == cod) & (todo["VARIABLE"] == var)] if len(todo) else pd.DataFrame()
        fila = {"COD_ESTACION": cod, "ESTACION": g["ESTACION"].iloc[0], "VARIABLE": var}
        if not len(t):
            fila.update({"ESTADO": "SIN DATOS" if (g["ESTADO"] == "SIN DATOS").all() else
                         "SIN RESPUESTA" if (g["ESTADO"] == "SIN RESPUESTA").any() else g["ESTADO"].iloc[0]})
            ctrl.append(fila); continue
        f = pd.to_datetime(t["FECHA"], format="%d/%m/%Y").sort_values()
        rango = (f.max() - f.min()).days + 1
        v = t.assign(_f=pd.to_datetime(t["FECHA"], format="%d/%m/%Y")).sort_values("_f")["VALOR"].astype(float)
        salto = v.diff().abs()
        lim = salto.quantile(0.9) if salto.notna().sum() > 10 else 0
        saltos = int((salto > 20 * lim).sum()) if lim and lim > 0 else 0     # saltos 20 veces mayores que lo usual
        fila.update({"PRIMERA_FECHA": f.min().strftime("%d/%m/%Y"), "ULTIMA_FECHA": f.max().strftime("%d/%m/%Y"),
                     "DIAS_CON_DATO": len(f), "DIAS_FALTANTES": rango - len(f), "SALTOS_RAROS": saltos,
                     "LLEGA_AL_INICIO": f.min().date() <= DESDE + timedelta(days=31)})
        fila["ESTADO"] = ("REVISAR (saltos)" if saltos else "OK" if fila["LLEGA_AL_INICIO"] else
                          f"OK desde {fila['PRIMERA_FECHA']} (SENAMHI no trae más atrás)")
        ctrl.append(fila)
ctrl = pd.DataFrame(ctrl)
if len(men) and len(todo) and len(ctrl):
    t = todo.copy()
    t["UNI"] = t["UNIDAD"].map(sin_tildes)
    prom = t.groupby(["COD_ESTACION", "UNI", "AÑO", "MES"])["VALOR"].agg(["mean", "count"]).reset_index()
    mm = men.assign(UNI=men["UNIDAD"].map(sin_tildes)).merge(prom, on=["COD_ESTACION", "UNI", "AÑO", "MES"])
    mm = mm[mm["count"] >= 25]
    if len(mm):
        mm["DIF_%"] = (mm["mean"] / mm["VALOR_MES"] - 1) * 100
        cruce = mm.groupby("COD_ESTACION")["DIF_%"].agg(lambda s: round(s.abs().max(), 2))
        ctrl["CRUCE_MENSUAL_DIF_MAX_%"] = ctrl["COD_ESTACION"].map(cruce)
ctrl["FECHA_EXTRACCION"] = HOY_TXT
guardar(ctrl, NOMBRE_CONTROL, DRIVE_FOLDER_CONTROL_ID, control_drive)

# ── 8. RESUMEN ───────────────────────────────────────────────────────────────
print()
print("Historia del hidrograma:", "✅ SENAMHI acepta fechas pasadas" if historia_ok else
      "⚠️ SENAMHI ignora la fecha pedida: solo trae los 2 últimos años hidrológicos" if pruebas_historia else
      "no se probó (el período elegido no lo necesitó)")
if len(ctrl):
    display(ctrl.assign(ESTADO=ctrl["ESTADO"].astype(str).str.replace(r" \(.*| desde .*", "", regex=True))
            .groupby(["VARIABLE", "ESTADO"]).size().to_frame("estaciones"))
    rev = ctrl[ctrl["ESTADO"].astype(str).str.startswith(("REVISAR", "SIN RESPUESTA"))]
    if len(rev):
        print(f"⚠️ A revisar: {len(rev)} (detalle en CONTROL/{NOMBRE_CONTROL}); primeros 15:")
        display(rev.head(15))
if len(todo):
    print("Días con dato por año (estaciones con nivel / caudal):")
    display(todo.groupby(["AÑO", "VARIABLE"])["COD_ESTACION"].nunique().unstack().fillna(0).astype(int))
    lor = todo[todo["COD_ESTACION"].map(DEPTO).fillna("") == "LORETO"]
    if len(lor):
        print("Loreto · último dato:")
        display(lor.assign(_f=pd.to_datetime(lor["FECHA"], format="%d/%m/%Y")).sort_values("_f")
                .groupby(["ESTACION", "VARIABLE"]).tail(1)[["ESTACION", "RIO", "VARIABLE", "FECHA", "VALOR", "UNIDAD",
                                                            "UMBRAL_ROJO", "UMBRAL_NARANJA", "UMBRAL_AMARILLO"]])
print(f"   Datos: https://drive.google.com/drive/folders/{DRIVE_FOLDER_ID}")
print(f"   Control: https://drive.google.com/drive/folders/{DRIVE_FOLDER_CONTROL_ID}")